In [ ]:
import sys
print(f"Python version: {sys.version}")
print(f"Python path: {sys.executable}")

Python version: 3.11.11 | packaged by conda-forge | (main, Dec  5 2024, 14:17:24) [GCC 13.3.0]
Python path: /opt/conda/bin/python3.11


In [ ]:
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")



PyTorch version: 2.6.0+cu124
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4080 Laptop GPU
CUDA memory allocated: 0.00 GB


In [ ]:

import numpy as np
import pandas as pd
import transformers
import sentence_transformers
import llama_index
import langchain

print(f"\nNumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Sentence Transformers: {sentence_transformers.__version__}")
print(f"LangChain: {langchain.__version__}")


NumPy: 1.26.4
Pandas: 2.2.1
Transformers: 4.40.2
Sentence Transformers: 3.1.1
LangChain: 1.2.3


In [ ]:
from sentence_transformers import SentenceTransformer
import os

model = SentenceTransformer('all-MiniLM-L6-v2')
model.save('models/embeddings_model/all_MiniLM_L6_v2/')

In [3]:
import httpx
import json
import numpy as np
import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer

# Test all services
print("Testing all services...")
services = {
    "LLM Server": "http://localhost:8001/health",
    "Qdrant": "http://localhost:6333"
}

for name, url in services.items():
    try:
        if name == "Qdrant":
            resp = httpx.get(url, timeout=5)
            status = "✓" if resp.status_code == 200 else "✗"
        else:
            resp = httpx.get(url, timeout=5)
            data = resp.json()
            status = "✓" if data.get("status") == "healthy" else "✗"
        print(f"{name}: {status}")
    except Exception as e:
        print(f"{name}: ✗ ({str(e)[:50]})")


Testing all services...
LLM Server: ✓
Qdrant: ✓


In [4]:
from src.customFunctions import *

In [5]:
collection_name="corpus"

In [6]:
client=loadfileDB(collection_name=collection_name, filepath='/workspace/data/fiqa/corpus.jsonl', colname='doc', chunk_size=1000)


Setting up Qdrant...
Collection already exists and has data. Skipping upload. 57638 points found.


In [7]:
test_query="Filing personal with 1099s versus business s-corp?"
results=retrieveQueryEmbeddings(query_texts=test_query,
                                client=client,collection_name=collection_name,colname='doc',k=3,verbose=True)
llmResponse=llmRespose(query_results=results, test_query=test_query, colname='doc', verbose=True)


1. Score: 0.689
   ID: 7826
   doc: ['I see a lot of people making the mistake or being given bad advise in structuring a new business. If you have more than one shareholder, then by all means an S Corporation is a better structure for lower taxes; avoid double taxation. If, however, this is a one shareholder S Corp, then you had better 1099 yourself as a consultant or look into sole proprietorship. The tax benefits are much better either way. Dr. Suraiya Shaik Ali']...

2. Score: 0.674
   ID: 3064
   doc: ["Depends whom the 1099 was issued to. If it was issued to your corporation - then its your corporation's income, not yours. Why would it go to your tax return? Your corporation and you are two separate legal entities. You will have to file the 1120S, whether you have corporate income or not, it has to be filed each year. So why make a mess of your reporting and not just report the corporation income on its return and your personal income on your own return? If you no longer use the

In [9]:
ground_truth = ["Depends whom the 1099 was issued to. If it was issued to your corporation - then its your corporation's income, not yours. Why would it go to your tax return? Your corporation and you are two separate legal entities. You will have to file the 1120S, whether you have corporate income or not, it has to be filed each year. So why make a mess of your reporting and not just report the corporation income on its return and your personal income on your own return? If you no longer use the corporation and all the 1099's are issued to you personally, then just dissolve it so that you won't have to file an empty 1120S every year and pay additional fees for maintaining it."]
# groundContext=df['retrieved_contexts'][1]
evaluate_answer(generated_answer=llmResponse, ground_truth=ground_truth, query_results=results, groundContext="",colname='doc')

/opt/conda/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'rouge1': 0.37815126050420167,
 'rouge2': 0.06779661016949151,
 'rougeL': 0.18487394957983191,
 'rougeLsum': 0.2352941176470588,
 'bertscore_precision': 0.8258732557296753,
 'bertscore_recall': 0.8436300754547119,
 'bertscore_f1': 0.8346572518348694,
 'bleu': 0.0,
 'cosine_similarity': 0.6378611326217651,
 'avg_retrieval_score': 0.6701138666666667,
 'avg_retrieval_similarity': 0.0}